# Nautilus Trader + Interactive Brokers: Quant Model Analysis

A production-shaped two-layer quant system for cryptocurrency (24/7) trading with **Nautilus Trader**, **Huber regression alpha**, and **Optuna hyperparameter optimization**.

## System Architecture

```
┌────────────────────────┐      yhat       ┌────────────────────┐     order objects
│  ML / ALPHA LAYER       │ ─────────────▶  │  STRATEGY LAYER     │ ──────────────────▶ IBKR
│  • PredictionEngine     │  fwd-return     │  • MLStrategy       │  LIMIT (maker) /
│  • Huber regression     │  forecast       │  • Risk Manager     │  MARKET (taker)
│  • Regime features      │                 │  • Position Manager │  (fractional coins)
│  • Cross-asset features │                 │                     │
└────────────────────────┘                 └────────────────────┘
```

**Key Features:**
- Walk-forward alpha model with no lookahead
- Regime awareness (Markov transition matrix + GaussianHMM)
- Cross-asset ARDL features
- 1% risk per trade, 0.25% hard cap, Kelly-sized when enabled
- Fractional Kelly conviction sizing
- Real IBKR bars with no synthetic lag


## Setup: Import Libraries and Load Data

In [ ]:
import sys
sys.path.insert(0, '/Users/lindawang/Documents/Workspace')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import logging
from datetime import datetime

# Configure plotting
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✓ Libraries imported successfully")

## Load IBKR Data

In [ ]:
# Load the real IBKR cryptocurrency data
data_path = Path('/Users/lindawang/Documents/Workspace/quant/data/ibkr_bars.csv')
df = pd.read_csv(data_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(f"Data shape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nUnique tickers: {sorted(df['symbol'].unique())}")
print(f"\nData summary:")
print(df.head(10))

## Data Overview by Ticker

In [ ]:
# Show data availability per ticker
ticker_stats = df.groupby('symbol').agg({
    'timestamp': ['min', 'max', 'count'],
    'close': ['mean', 'std', 'min', 'max']
}).round(2)

print("Data availability per ticker:")
print(ticker_stats)

# Calculate returns per ticker
df['log_return'] = df.groupby('symbol')['close'].transform(lambda x: np.log(x / x.shift(1)))

print("\n\nReturn statistics by ticker:")
return_stats = df.groupby('symbol')['log_return'].agg([
    ('count', 'count'),
    ('mean_%', lambda x: x.mean() * 100),
    ('std_%', lambda x: x.std() * 100),
    ('min_%', lambda x: x.min() * 100),
    ('max_%', lambda x: x.max() * 100),
]).round(2)
print(return_stats)

## Optimize Parameters Summary

In [ ]:
# Load Optuna optimization results
params_path = Path('/Users/lindawang/Documents/Workspace/quant/optimize/best_params.json')
with open(params_path) as f:
    opt_results = json.load(f)

print("\n📊 OPTIMIZATION RESULTS")
print("=" * 60)

params = opt_results['params']
print(f"\nBest in-sample optimization score: {opt_results['in_sample_value']:.4f}")
print(f"Training fraction: {opt_results['train_frac']:.2%}")
print(f"Random seed: {opt_results['seed']}")
print(f"Trials: {opt_results['trials']}")

print("\n📋 TUNED HYPERPARAMETERS:")
print("-" * 60)
for key, value in params.items():
    if isinstance(value, float):
        print(f"  {key:.<40} {value:.6f}")
    else:
        print(f"  {key:.<40} {value}")

print("\n🔧 STRUCTURAL SETTINGS (FIXED):")
print("-" * 60)
for key in ['refit_every_n_bars', 'warmup_bars', 'min_train_bars']:
    if key in opt_results:
        print(f"  {key:.<40} {opt_results[key]}")

## Run Backtest with Optimized Parameters

In [ ]:
from quant.run.backtest_common import build_and_run, VENUE
from quant.run.metrics import compute_metrics, render_panel
from quant.run.run_backtest import load_best_params

print("\n🚀 RUNNING BACKTEST WITH OPTIMIZED PARAMETERS")
print("=" * 60)

csv_path = 'quant/data/ibkr_bars.csv'
tickers = ['BTC', 'ETH', 'SOL', 'XRP', 'DOGE']
starting_cash = 5000.0

# Load the optimized parameters
params_path = 'quant/optimize/best_params.json'
overrides = load_best_params(params_path)
print(f"\nLoaded {len(overrides)} parameter overrides")

# Run the backtest
engine = build_and_run(
    csv_path=csv_path,
    tickers=tickers,
    strategy_overrides=overrides,
    starting_cash=starting_cash,
    log_level='WARNING',
    asset_class='crypto'
)

print("\n✓ Backtest completed successfully")

## Strategy Performance Metrics

In [ ]:
# Compute and display metrics
metrics = compute_metrics(engine, VENUE, starting_cash=starting_cash)

# Display rich metrics panel
print(render_panel(metrics))

# Store metrics as dict for later analysis
metrics_dict = metrics.as_dict()
print("\n📊 Metrics as dictionary:")
for key, value in metrics_dict.items():
    print(f"  {key:.<35} {value}")

## Equity Curve & Drawdown Analysis

In [ ]:
# Extract equity curve from the account report
try:
    account_report = engine.trader.generate_account_report(VENUE)
    
    # Try to find the equity column
    equity_col = None
    for col in ['total', 'balance_total', 'free', 'equity']:
        if col in account_report.columns:
            equity_col = col
            break
    
    if equity_col:
        # Extract and clean equity values
        equity_series = account_report[equity_col].astype(str).str.replace(r'[^0-9.\-eE]', '', regex=True)
        equity_values = pd.to_numeric(equity_series, errors='coerce').dropna().values
        
        if len(equity_values) > 1:
            # Calculate drawdown
            running_max = np.maximum.accumulate(equity_values)
            drawdown = (equity_values - running_max) / running_max * 100
            
            # Plot equity curve and drawdown
            fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
            
            # Equity curve
            axes[0].plot(equity_values, linewidth=2, color='steelblue', label='Equity')
            axes[0].axhline(starting_cash, color='gray', linestyle='--', alpha=0.5, label=f'Starting Cash (${starting_cash:,.0f})')
            axes[0].fill_between(range(len(equity_values)), starting_cash, equity_values, alpha=0.3, color='steelblue')
            axes[0].set_ylabel('Equity ($)', fontsize=11, fontweight='bold')
            axes[0].set_title('Equity Curve Over Time', fontsize=12, fontweight='bold')
            axes[0].legend(loc='best')
            axes[0].grid(True, alpha=0.3)
            
            # Drawdown
            axes[1].fill_between(range(len(drawdown)), drawdown, 0, alpha=0.5, color='red')
            axes[1].plot(drawdown, linewidth=1.5, color='darkred')
            axes[1].set_ylabel('Drawdown (%)', fontsize=11, fontweight='bold')
            axes[1].set_xlabel('Trading Day', fontsize=11, fontweight='bold')
            axes[1].set_title('Peak-to-Trough Drawdown', fontsize=12, fontweight='bold')
            axes[1].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
            
            print(f"✓ Equity curve plotted ({len(equity_values)} data points)")
    else:
        print("⚠ Could not find equity column in account report")
except Exception as e:
    print(f"⚠ Could not extract equity curve: {e}")

## Trade Analysis & Win Rate

In [ ]:
# Extract positions report to analyze trades
try:
    positions = engine.trader.generate_positions_report()
    
    if positions is not None and len(positions) > 0:
        # Filter to snapshot rows (actual closed trades)
        if 'is_snapshot' in positions.columns:
            trades = positions[positions['is_snapshot'] == True].copy()
        else:
            trades = positions.copy()
        
        # Extract PnL column
        pnl_col = None
        for col in ['realized_pnl', 'realised_pnl', 'pnl_realized', 'pnl']:
            if col in trades.columns:
                pnl_col = col
                break
        
        if pnl_col:
            # Clean PnL values
            pnl_series = trades[pnl_col].astype(str).str.replace(r'[^0-9.\-eE]', '', regex=True)
            pnl_values = pd.to_numeric(pnl_series, errors='coerce').dropna().values
            
            if len(pnl_values) > 0:
                # Calculate trade statistics
                winners = pnl_values[pnl_values > 0]
                losers = pnl_values[pnl_values < 0]
                
                print("\n💹 TRADE ANALYSIS")
                print("=" * 60)
                print(f"Total trades closed: {len(pnl_values)}")
                print(f"Winning trades:      {len(winners)}")
                print(f"Losing trades:       {len(losers)}")
                print(f"Breakeven trades:    {len(pnl_values) - len(winners) - len(losers)}")
                print(f"\nWin rate:            {100 * len(winners) / len(pnl_values):.1f}%")
                
                if len(winners) > 0 and len(losers) > 0:
                    print(f"\nAvg winning trade:   ${winners.mean():.2f}")
                    print(f"Avg losing trade:    ${losers.mean():.2f}")
                    print(f"Win/Loss ratio:      {abs(winners.mean() / losers.mean()):.2f}x")
                    print(f"\nGross profit:        ${winners.sum():.2f}")
                    print(f"Gross loss:          ${losers.sum():.2f}")
                    if losers.sum() != 0:
                        profit_factor = abs(winners.sum() / losers.sum())
                        print(f"Profit factor:       {profit_factor:.2f}x")
                
                # Plot PnL distribution
                fig, axes = plt.subplots(1, 2, figsize=(14, 5))
                
                # PnL histogram
                axes[0].hist(pnl_values, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
                axes[0].axvline(0, color='red', linestyle='--', linewidth=2, label='Breakeven')
                axes[0].axvline(pnl_values.mean(), color='green', linestyle='--', linewidth=2, label=f'Mean PnL: ${pnl_values.mean():.2f}')
                axes[0].set_xlabel('Trade PnL ($)', fontweight='bold')
                axes[0].set_ylabel('Frequency', fontweight='bold')
                axes[0].set_title('Distribution of Trade Outcomes', fontweight='bold')
                axes[0].legend()
                axes[0].grid(True, alpha=0.3)
                
                # Cumulative PnL
                cumulative_pnl = np.cumsum(pnl_values)
                axes[1].plot(cumulative_pnl, linewidth=2, color='steelblue')
                axes[1].fill_between(range(len(cumulative_pnl)), cumulative_pnl, 0, alpha=0.3, color='steelblue')
                axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)
                axes[1].set_xlabel('Trade Number', fontweight='bold')
                axes[1].set_ylabel('Cumulative PnL ($)', fontweight='bold')
                axes[1].set_title('Cumulative P&L Over Time', fontweight='bold')
                axes[1].grid(True, alpha=0.3)
                
                plt.tight_layout()
                plt.show()
    else:
        print("⚠ No closed positions/trades found")
except Exception as e:
    print(f"⚠ Could not extract trade analysis: {e}")

## Risk Analysis: Daily Loss & Drawdown

In [ ]:
print("\n⚠️  RISK ANALYSIS")
print("=" * 60)

print("\n📋 Risk Rules (Hard-Coded):")
risk_rules = [
    ('Risk budget / trade', '1%', 'of equity'),
    ('Hard cap / trade', '0.25%', 'of equity'),
    ('Leverage (per trade)', '1.0', 'max notional / equity'),
    ('Leverage (book)', '1.0', 'aggregate notional / equity'),
    ('Daily loss limit', '2%', 'flatten all + 24h halt'),
    ('Drawdown warning', '5%', 'peak-to-trough'),
    ('Kill-switch', '10%', 'peak-to-trough (permanent)'),
    ('Kelly ceiling', '50%', 'of equity notional'),
]

for rule, value, context in risk_rules:
    print(f"  {rule:.<30} {value:>10}  {context}")

print(f"\n💰 Starting Capital: ${starting_cash:,.2f}")

# From metrics
print(f"\n📊 Observed Metrics:")
print(f"  Net Profit: ${metrics.net_profit_usd:>15,.2f}")
print(f"  Max Drawdown: {metrics.max_drawdown_pct:>13.2f}%")
print(f"  Sharpe Ratio: {metrics.sharpe_ratio:>14.2f}")
print(f"  Turnover: {metrics.turnover_rate:>20.2f}x")

## Cross-Asset Features (ARDL + Spread Lags)

In [ ]:
print("\n🔗 CROSS-ASSET FEATURES")
print("=" * 60)

print("\nCross-asset lags enabled: ", params.get('cross_asset_lags', 0) > 0)
print("Spread lags enabled: ", params.get('spread_lags', 0) > 0)

if params.get('spread_lags', 0) > 0:
    print(f"\n✓ Spread lag features enabled (lag={params['spread_lags']})")
    print("  Each instrument's Huber model can condition on:")
    print("  - Other tickers' lagged log returns (ARDL)")
    print("  - Spread (own_return - peer_return) lagged values")
    print("  - Mean-reverting divergence captured as regression input")
else:
    print("\n✗ Cross-asset features disabled in optimized parameters")

# Show correlation matrix for universe
print("\n\n📊 Return Correlation Matrix (Universe):")
print("-" * 60)

correlation_data = {}
for ticker in ['BTC', 'ETH', 'SOL', 'XRP', 'DOGE']:
    ticker_df = df[df['symbol'] == ticker].sort_values('timestamp')
    if len(ticker_df) > 0:
        correlation_data[ticker] = ticker_df['log_return'].values

# Create correlation matrix
corr_df = pd.DataFrame(correlation_data).corr()
print(corr_df.round(3))

# Plot correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(corr_df, annot=True, fmt='.3f', cmap='coolwarm', center=0, 
            square=True, cbar_kws={'label': 'Correlation'}, vmin=-1, vmax=1)
plt.title('Log-Return Correlation Matrix (Crypto Universe)', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

## Summary & Interpretation

In [ ]:
print("\n" + "="*70)
print("🎯 BACKTEST SUMMARY & INTERPRETATION")
print("="*70)

print("\n1️⃣  BACKTEST RESULTS:")
print("-" * 70)
print(f"   Strategy: Two-layer (alpha + strategy) on {len(tickers)} crypto assets")
print(f"   Period: {df['timestamp'].min().date()} to {df['timestamp'].max().date()}")
print(f"   Starting capital: ${starting_cash:,.2f}")
print(f"   Final equity: ${starting_cash + metrics.net_profit_usd:,.2f}")
print(f"   Net return: {100 * metrics.net_profit_usd / starting_cash:.1f}%")

print(f"\n2️⃣  RISK-ADJUSTED PERFORMANCE:")
print("-" * 70)
print(f"   Sharpe ratio: {metrics.sharpe_ratio:.2f} (risk-adjusted return)")
print(f"   Max drawdown: {metrics.max_drawdown_pct:.2f}%")
print(f"   Recovery: Equity came back from {metrics.max_drawdown_pct:.2f}% loss")

print(f"\n3️⃣  TRADING ACTIVITY:")
print("-" * 70)
print(f"   Total fills (trades): {metrics.total_trades}")
print(f"   Closed positions: {metrics.closed_positions}")
print(f"   Turnover rate: {metrics.turnover_rate:.2f}x avg capital")
print(f"   Capacity score: ${metrics.capacity_score:.4f} profit per $1 traded")

print(f"\n4️⃣  WIN/LOSS STATISTICS:")
print("-" * 70)
print(f"   Win rate: {metrics.win_rate_pct:.1f}% ({metrics.wins} wins)")
print(f"   Losing trades: {metrics.losses}")
print(f"   Win/Loss ratio: {metrics.win_loss_ratio:.2f}x")
print(f"   Profit factor: {metrics.profit_factor}")

print(f"\n5️⃣  ALPHA LAYER:")
print("-" * 70)
print(f"   Model: Huber regression (robust to outliers)")
print(f"   Tuned alpha: {params['huber_alpha']:.3f}")
print(f"   Lags (AR): {params['n_lags']}")
print(f"   Forecast horizon: {params['horizon']} bars")
print(f"   Cross-asset lags: {params['cross_asset_lags']}")
print(f"   Regime-aware: Yes (Markov + GaussianHMM)")

print(f"\n6️⃣  STRATEGY LAYER:")
print("-" * 70)
print(f"   Entry signal: yhat > threshold ({params['entry_threshold']:.6f})")
print(f"   Stop loss: ATR-based (period={params['atr_period']}, mult={params['atr_stop_mult']:.2f})")
print(f"   Order type: {'LIMIT' if params['use_limit_orders'] else 'MARKET'}")
if params['use_limit_orders']:
    print(f"   Limit offset: {params['limit_offset_bps']:.2f} bps")
print(f"   Sizing: {'Kelly-fractioned' if params['use_kelly_sizing'] else 'Fixed (1% / 0.25% cap)'}")
print(f"   Max open positions: {params['max_open_positions']}")
print(f"   Risk per trade: 1% target, 0.25% hard cap")

print(f"\n7️⃣  OPTIMIZATION DETAILS:")
print("-" * 70)
print(f"   Method: Optuna (Bayesian hyperparameter search)")
print(f"   In-sample score: {opt_results['in_sample_value']:.4f}")
print(f"   Training fraction: {opt_results['train_frac']:.0%}")
print(f"   Trials: {opt_results['trials']}")
print(f"   Seed: {opt_results['seed']}")

print("\n" + "="*70)
print("\n✅ Analysis Complete!")

In [ ]:
# Cleanup
engine.dispose()
print("\n🧹 Engine cleaned up.")